# R1-U2 — AST wrapper and T4 feasibility probe

Use a **GPU runtime (T4)**. Run all cells top to bottom (about 10-15 minutes; downloads ~350 MB). Needs the project folder on Drive with
`data/splits/` and `data/processed/chunks_v1.parquet` (for the workload) and the latest `reports/r1/r1_u1_colab_*.json` (WavLM speed).

It produces `reports/r1/ast_probe.json` with:
1. the loading audit (only the new task-head tensors may be new or reshaped), the module names, and trainable names/counts per stage;
2. proof that gradients and weight updates reach only the unfrozen parts;
3. fp16 forward+backward at microbatch 2/4/8 for the `head` and `partial` stages: clips/s (CPU feature step timed separately), peak VRAM,
   out-of-memory cases;
4. forward-only (evaluation) throughput;
5. epoch-time estimates and a **GPU-hours budget for all 8 final fits plus extraction, with a 25% reserve**; measured and assumed lines are labelled.

In [1]:
import os, sys, json, subprocess, zipfile, time, hashlib, platform, gc
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

ROOT = Path(os.environ.get("ECHOPROOF_ROOT", "/content/drive/MyDrive/PAI&AIES Project"))
assert ROOT.exists(), f"project folder not found: {ROOT}"

ZIP = ROOT / "echoproof_sync.zip"          # code only: splits, manifests and reports are never overwritten
if ZIP.exists():
    with zipfile.ZipFile(ZIP) as z:
        members = [m for m in z.namelist() if m.split("/")[0] in ("src", "scripts", "tests") and "__pycache__" not in m]
        z.extractall(ROOT, members)
    print(f"unpacked {len(members)} code files from {ZIP.name}")
SOURCES = {"ast_detector.py": ROOT / "src/echoproof/models/ast_detector.py", "gpu_budget.py": ROOT / "src/echoproof/gpu_budget.py",
           "pretrained.py": ROOT / "src/echoproof/features/pretrained.py", "test_r1_u2_ast.py": ROOT / "tests/test_r1_u2_ast.py"}
missing = [k for k, p in SOURCES.items() if not p.exists()]
assert not missing, f"missing in the sync zip: {missing} - rebuild echoproof_sync.zip and upload it"
SHA = {k: hashlib.sha256(p.read_bytes()).hexdigest() for k, p in SOURCES.items()}
for need in ("data/splits/protocol_A_v1.parquet", "data/splits/protocol_B_v1.parquet", "data/processed/chunks_v1.parquet"):
    assert (ROOT / need).exists(), f"missing {need} (needed for the workload)"
print(json.dumps({k: v[:12] for k, v in SHA.items()}, indent=2))

Mounted at /content/drive
{
  "ast_detector.py": "705a52fdf633",
  "gpu_budget.py": "b32f23c70bf3",
  "pretrained.py": "46aa840888db",
  "test_r1_u2_ast.py": "5738c43acbc4"
}


In [2]:
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
sys.dont_write_bytecode = True
sys.path.insert(0, str(ROOT / "src"))

import numpy as np, torch, transformers, huggingface_hub
import torch.nn.functional as F
try:
    import torchaudio
    TORCHAUDIO = torchaudio.__version__
except ImportError:
    TORCHAUDIO = None
CUDA = torch.cuda.is_available()
DEVICE = "cuda" if CUDA else "cpu"
ENV = {"python": platform.python_version(), "torch": torch.__version__, "transformers": transformers.__version__, "torchaudio": TORCHAUDIO,
       "huggingface_hub": huggingface_hub.__version__, "cuda": CUDA, "gpu": torch.cuda.get_device_name(0) if CUDA else None,
       "vram_gb": round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1) if CUDA else None, "cpu_count": os.cpu_count()}
print(json.dumps(ENV, indent=2))
assert CUDA, "this probe needs the T4 GPU runtime (Runtime > Change runtime type)"

{
  "python": "3.13.16",
  "torch": "2.11.0+cu130",
  "transformers": "5.22.0",
  "torchaudio": "2.11.0+cu130",
  "huggingface_hub": "1.33.0",
  "cuda": true,
  "gpu": "Tesla T4",
  "vram_gb": 14.6,
  "cpu_count": 2
}


## 1. Unit tests on this runtime

In [3]:
proc = subprocess.run([sys.executable, "-m", "pytest", "tests/test_r1_u2_ast.py", "-q", "-p", "no:cacheprovider"],
                      cwd=ROOT, env={**os.environ, "PYTHONPATH": str(ROOT / "src")}, capture_output=True, text=True)
print(proc.stdout[-2500:] + proc.stderr[-1500:])
UNIT_TESTS_PASSED = proc.returncode == 0
PYTEST_SUMMARY = next((l.strip() for l in reversed(proc.stdout.splitlines()) if " passed" in l or " failed" in l or " error" in l), "no summary line")
print("unit tests:", "PASSED" if UNIT_TESTS_PASSED else "FAILED", "|", PYTEST_SUMMARY)

...............                                                          [100%]
15 passed in 21.20s

unit tests: PASSED | 15 passed in 21.20s


## 2. Pinned download, loading audit, stages

In [4]:
from echoproof.features.pretrained import PRETRAINED, file_sha256, pinned_snapshot
from echoproof.models.ast_detector import ASTDetector, HEAD_TENSORS

path = pinned_snapshot("ast")
weights = path / PRETRAINED["ast"]["weights_file"]
WEIGHTS = {"revision": path.name, "pinned_revision": PRETRAINED["ast"]["revision"], "sha256": file_sha256(weights), "bytes": weights.stat().st_size,
           "recorded_sha256_matches": (file_sha256(weights) == PRETRAINED["ast"]["weights_sha256"]) if PRETRAINED["ast"]["weights_sha256"] else "none recorded yet - copy the value into PRETRAINED"}
print(json.dumps(WEIGHTS, indent=1))

pre = json.loads((path / "preprocessor_config.json").read_text())
PREPROCESSOR = {k: pre.get(k) for k in ("feature_extractor_type", "sampling_rate", "num_mel_bins", "max_length", "do_normalize", "mean", "std")}
print("preprocessor:", json.dumps(PREPROCESSOR))

det = ASTDetector.from_pinned(device=DEVICE, seed=42)
LOADING = det.loading
print("loading audit:", json.dumps(LOADING, indent=1))
print("allowed new/reshaped tensors:", HEAD_TENSORS)

REPORTS = {}
for stage in ("frozen", "head", "partial"):
    det.set_stage(stage, n_blocks=2)
    rep = det.parameter_report()
    REPORTS[stage] = rep
    print(f"--- stage={stage}: trainable {rep['trainable_parameters']:,} of {rep['total_parameters']:,} ({100 * rep['trainable_fraction']:.2f}%)")
    for g, v in rep["groups"].items():
        print(f"    {g:<12} {v['trainable']:>11,} / {v['parameters']:>11,}  ({v['tensors']} tensors)")
    if stage != "frozen":
        print("    trainable tensors:", len(rep["trainable_names"]), "| first:", rep["trainable_names"][:2], "| last:", rep["trainable_names"][-2:])

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

{
 "revision": "f826b80d28226b62986cc218e5cec390b1096902",
 "pinned_revision": "f826b80d28226b62986cc218e5cec390b1096902",
 "sha256": "ae0c1e2ad4e1381d851fa9bf298ba13ebc9c5a914cdee2dbe427a6583869924d",
 "bytes": 346404948,
 "recorded_sha256_matches": true
}
preprocessor: {"feature_extractor_type": "ASTFeatureExtractor", "sampling_rate": 16000, "num_mel_bins": 128, "max_length": 1024, "do_normalize": true, "mean": -4.2677393, "std": 4.5689974}


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `527`.


Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

[transformers] ASTForAudioClassification LOAD REPORT from: /root/.cache/huggingface/hub/models--MIT--ast-finetuned-audioset-10-10-0.4593/snapshots/f826b80d28226b62986cc218e5cec390b1096902
Key                     | Status   |                                                                                         
------------------------+----------+-----------------------------------------------------------------------------------------
classifier.dense.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([527]) vs model:torch.Size([2])          
classifier.dense.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([527, 768]) vs model:torch.Size([2, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


loading audit: {
 "missing": [],
 "unexpected": [],
 "mismatched": [
  "classifier.dense.bias",
  "classifier.dense.weight"
 ],
 "ok": true,
 "non_head_new_or_reshaped": [],
 "source": "transformers loading info + shape audit",
 "library_info_available": true,
 "shapes": {
  "ok": true,
  "model_only_shapes": {
   "(2, 768)": 1,
   "(2,)": 1
  },
  "checkpoint_only_shapes": {
   "(527,)": 1,
   "(527, 768)": 1
  },
  "tensors_in_model": 203,
  "tensors_in_checkpoint": 203
 }
}
allowed new/reshaped tensors: ('classifier.dense.weight', 'classifier.dense.bias')
--- stage=frozen: trainable 0 of 86,190,338 (0.00%)
    block.00               0 /   7,087,872  (16 tensors)
    block.01               0 /   7,087,872  (16 tensors)
    block.02               0 /   7,087,872  (16 tensors)
    block.03               0 /   7,087,872  (16 tensors)
    block.04               0 /   7,087,872  (16 tensors)
    block.05               0 /   7,087,872  (16 tensors)
    block.06               0 /   7,087,87

## 3. Gradients and updates reach only the unfrozen parts (real model, fp32, one step)

In [5]:
det.set_stage("partial", n_blocks=2)
det.train()
wave = torch.randn(2, 64000, device=DEVICE) * 0.05
valid = torch.full((2,), 64000, dtype=torch.long)
opt = torch.optim.AdamW(det.param_groups(head_lr=1e-3, backbone_lr=1e-3), weight_decay=0.0)
before = det.digests()
loss = F.cross_entropy(det(wave, valid), torch.tensor([0, 1], device=DEVICE))
loss.backward()
expected = {"classifier", "final_norm", "block.10", "block.11"}
grad_groups = {}
for name, p in det.model.named_parameters():
    g = det._group(name)
    state = grad_groups.setdefault(g, {"with_grad": 0, "nonzero": 0, "tensors": 0})
    state["tensors"] += 1
    if p.grad is not None:
        state["with_grad"] += 1
        state["nonzero"] += int(float(p.grad.abs().sum()) > 0)
opt.step(); opt.zero_grad(set_to_none=True)
after = det.digests()
changed = sorted(g for g in before if before[g] != after[g])
GRADIENT_PROOF = {"expected_updated_groups": sorted(expected), "groups_with_gradients": sorted(g for g, v in grad_groups.items() if v["with_grad"]),
                  "groups_changed_by_one_step": changed, "frozen_groups_bit_identical": sorted(g for g in before if before[g] == after[g]),
                  "per_group": grad_groups, "ok": set(changed) == expected and {g for g, v in grad_groups.items() if v["with_grad"]} == expected
                  and all(grad_groups[g]["nonzero"] > 0 for g in expected)}
print(json.dumps({k: v for k, v in GRADIENT_PROOF.items() if k != "per_group"}, indent=1))
del opt; gc.collect(); torch.cuda.empty_cache()

{
 "expected_updated_groups": [
  "block.10",
  "block.11",
  "classifier",
  "final_norm"
 ],
 "groups_with_gradients": [
  "block.10",
  "block.11",
  "classifier",
  "final_norm"
 ],
 "groups_changed_by_one_step": [
  "block.10",
  "block.11",
  "classifier",
  "final_norm"
 ],
 "frozen_groups_bit_identical": [
  "block.00",
  "block.01",
  "block.02",
  "block.03",
  "block.04",
  "block.05",
  "block.06",
  "block.07",
  "block.08",
  "block.09",
  "embeddings"
 ],
 "ok": true
}


## 4. fp16 training probe: microbatch 2 / 4 / 8, `head` and `partial` stages

Per case the CPU feature step (AST filterbank, done inside the forward pass) is timed separately from the GPU step. `serial` is what the
current wrapper does (extract, then compute); `overlapped` is the best case if features were prepared in DataLoader workers.

In [6]:
def probe(stage, mb, steps=8, warmup=2):
    result = {"stage": stage, "microbatch": mb, "status": "ok"}
    det.set_stage(stage, n_blocks=2)
    wave = torch.randn(mb, 64000, device=DEVICE) * 0.05
    valid = torch.full((mb,), 64000, dtype=torch.long)
    labels = torch.arange(mb, device=DEVICE) % 2
    times = []
    for _ in range(5):
        t = time.perf_counter(); feats = det.extract_features(wave, valid); times.append(time.perf_counter() - t)
    extract_s = float(np.median(times))
    feats = feats.to(DEVICE)
    opt = scaler = None
    try:
        opt = torch.optim.AdamW(det.param_groups(head_lr=1e-4, backbone_lr=1e-5), weight_decay=0.01)
        trainable = [p for g in opt.param_groups for p in g["params"]]
        scaler = torch.amp.GradScaler("cuda")
        det.train()
        gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
        times = []
        for i in range(warmup + steps):
            torch.cuda.synchronize(); t = time.perf_counter()
            with torch.autocast("cuda", dtype=torch.float16):
                logits = det.model(input_values=feats).logits
            loss = F.cross_entropy(logits.float(), labels)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(trainable, 1.0)
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
            torch.cuda.synchronize(); times.append(time.perf_counter() - t)
        gpu_s = float(np.median(times[warmup:]))
        result.update(extract_ms_per_clip=round(extract_s / mb * 1000, 1), gpu_ms_per_clip=round(gpu_s / mb * 1000, 1),
                      serial_clips_per_s=round(mb / (extract_s + gpu_s), 2), overlapped_clips_per_s=round(mb / max(extract_s, gpu_s), 2),
                      peak_vram_mb=round(torch.cuda.max_memory_allocated() / 2**20), loss_finite=bool(torch.isfinite(loss).item()))
    except torch.cuda.OutOfMemoryError:
        result.update(status="oom", peak_vram_mb=round(torch.cuda.max_memory_allocated() / 2**20))
    finally:
        del opt, scaler
        gc.collect(); torch.cuda.empty_cache()
    return result

PROBES = []
for stage in ("head", "partial"):
    for mb in (2, 4, 8):
        r = probe(stage, mb)
        PROBES.append(r)
        print(json.dumps(r))

{"stage": "head", "microbatch": 2, "status": "ok", "extract_ms_per_clip": 4.0, "gpu_ms_per_clip": 18.0, "serial_clips_per_s": 45.45, "overlapped_clips_per_s": 55.45, "peak_vram_mb": 406, "loss_finite": true}
{"stage": "head", "microbatch": 4, "status": "ok", "extract_ms_per_clip": 3.7, "gpu_ms_per_clip": 17.1, "serial_clips_per_s": 48.18, "overlapped_clips_per_s": 58.59, "peak_vram_mb": 465, "loss_finite": true}
{"stage": "head", "microbatch": 8, "status": "ok", "extract_ms_per_clip": 5.2, "gpu_ms_per_clip": 16.7, "serial_clips_per_s": 45.69, "overlapped_clips_per_s": 59.85, "peak_vram_mb": 582, "loss_finite": true}
{"stage": "partial", "microbatch": 2, "status": "ok", "extract_ms_per_clip": 5.2, "gpu_ms_per_clip": 27.9, "serial_clips_per_s": 30.29, "overlapped_clips_per_s": 35.89, "peak_vram_mb": 680, "loss_finite": true}
{"stage": "partial", "microbatch": 4, "status": "ok", "extract_ms_per_clip": 3.8, "gpu_ms_per_clip": 25.0, "serial_clips_per_s": 34.77, "overlapped_clips_per_s": 40.

## 5. Forward-only throughput (validation / test scoring)

In [7]:
EVAL = []
det.set_stage("frozen")
det.eval()
for mb in (16, 32):
    wave = torch.randn(mb, 64000, device=DEVICE) * 0.05
    valid = torch.full((mb,), 64000, dtype=torch.long)
    times = []
    for _ in range(3):
        t = time.perf_counter(); feats = det.extract_features(wave, valid); times.append(time.perf_counter() - t)
    extract_s = float(np.median(times))
    feats = feats.to(DEVICE)
    try:
        gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
        times = []
        with torch.inference_mode(), torch.autocast("cuda", dtype=torch.float16):
            for i in range(6):
                torch.cuda.synchronize(); t = time.perf_counter()
                det.model(input_values=feats)
                torch.cuda.synchronize(); times.append(time.perf_counter() - t)
        gpu_s = float(np.median(times[1:]))
        row = {"microbatch": mb, "status": "ok", "extract_ms_per_clip": round(extract_s / mb * 1000, 1), "gpu_ms_per_clip": round(gpu_s / mb * 1000, 1),
               "serial_clips_per_s": round(mb / (extract_s + gpu_s), 2), "overlapped_clips_per_s": round(mb / max(extract_s, gpu_s), 2),
               "peak_vram_mb": round(torch.cuda.max_memory_allocated() / 2**20)}
    except torch.cuda.OutOfMemoryError:
        row = {"microbatch": mb, "status": "oom"}
        gc.collect(); torch.cuda.empty_cache()
    EVAL.append(row)
    print(json.dumps(row))

{"microbatch": 16, "status": "ok", "extract_ms_per_clip": 3.7, "gpu_ms_per_clip": 16.6, "serial_clips_per_s": 49.35, "overlapped_clips_per_s": 60.38, "peak_vram_mb": 815}
{"microbatch": 32, "status": "ok", "extract_ms_per_clip": 4.3, "gpu_ms_per_clip": 16.5, "serial_clips_per_s": 47.96, "overlapped_clips_per_s": 60.6, "peak_vram_mb": 1283}


## 6. Workload, recommended settings and the GPU-hours budget

In [8]:
from echoproof.gpu_budget import DEFAULT_PLAN, estimate_budget, workload_from_manifests

WORKLOAD = workload_from_manifests(ROOT / "data/splits", ROOT / "data/processed/chunks_v1.parquet")
print(json.dumps(WORKLOAD, indent=1))

u1_reports = sorted((ROOT / "reports/r1").glob("r1_u1_colab_*.json"))
assert u1_reports, "run the R1-U1 notebook first: its WavLM speed is an input to the budget"
WAVLM_CPS = json.loads(u1_reports[-1].read_text())["timing"]["clips_per_second"]
print("WavLM clips/s from", u1_reports[-1].name, "=", WAVLM_CPS)

limit_mb = 0.85 * torch.cuda.get_device_properties(0).total_memory / 2**20
def best(rows, key):
    ok = [r for r in rows if r["status"] == "ok" and r["peak_vram_mb"] <= limit_mb]
    return max(ok, key=lambda r: r[key]) if ok else None

best_head, best_partial = best([p for p in PROBES if p["stage"] == "head"], "serial_clips_per_s"), best([p for p in PROBES if p["stage"] == "partial"], "serial_clips_per_s")
best_eval = best(EVAL, "serial_clips_per_s")
assert best_head and best_partial and best_eval, "no probe case fit in memory; see the table above"
effective_batch = 16
RECOMMENDED = {"vram_limit_mb": round(limit_mb), "precision": "fp16 autocast + GradScaler (T4 has no fast bf16)",
                "head_stage_microbatch": best_head["microbatch"], "partial_stage_microbatch": best_partial["microbatch"],
                "accumulation_steps_for_effective_batch_16": max(1, effective_batch // best_partial["microbatch"]), "eval_microbatch": best_eval["microbatch"]}
print(json.dumps(RECOMMENDED, indent=1))

def budget(kind):
    key = f"{kind}_clips_per_s"
    measured = {"ast_head_train_cps": best_head[key], "ast_partial_train_cps": best_partial[key], "ast_eval_cps": best_eval[key], "wavlm_cps": WAVLM_CPS}
    return measured, estimate_budget(measured, WORKLOAD)

MEASURED_SERIAL, BUDGET_SERIAL = budget("serial")
MEASURED_OVERLAP, BUDGET_OVERLAP = budget("overlapped")
for label, b in (("SERIAL feature extraction (current wrapper)", BUDGET_SERIAL), ("OVERLAPPED (features prepared in DataLoader workers)", BUDGET_OVERLAP)):
    print(f"=== {label}")
    for i in b["items"]:
        print(f"  {i['item']:<46}{i['protocol']:<5}{i['hours']:>7.2f} h  [{i['basis']}]  {i['detail']}")
    print(f"  subtotal {b['subtotal_hours']} h (measured {b['measured_hours']} h, assumed {b['assumed_hours']} h) | +25% reserve = {b['total_hours_with_reserve']} h"
          f" | {b['sessions_needed']} sessions of {b['session_hours']} h | {b['hours_per_account']} h per account ({b['accounts']} accounts)")

# epoch estimates (single protocol, one AST fit)
EPOCH = {}
for protocol in ("B", "A"):
    w = WORKLOAD[protocol]
    train = w["train_recordings"] / MEASURED_SERIAL["ast_partial_train_cps"] / 60
    val = w["selection_chunks"] / MEASURED_SERIAL["ast_eval_cps"] / 60
    EPOCH[protocol] = {"partial_epoch_train_minutes": round(train, 1), "selection_validation_minutes": round(val, 1), "partial_epoch_total_minutes": round(train + val, 1),
                       "head_epoch_train_minutes": round(w["train_recordings"] / MEASURED_SERIAL["ast_head_train_cps"] / 60, 1)}
print(json.dumps(EPOCH, indent=1))

{
 "A": {
  "train_recordings": 7550,
  "train_chunks": 37123,
  "selection_chunks": 12331,
  "calibration_chunks": 5777,
  "operating_chunks": 6304,
  "test_chunks": 25547
 },
 "B": {
  "train_recordings": 4530,
  "train_chunks": 23096,
  "selection_chunks": 10029,
  "calibration_chunks": 4712,
  "operating_chunks": 5135,
  "test_chunks": 20558
 }
}
WavLM clips/s from r1_u1_colab_20261104T135216Z.json = 36.1
{
 "vram_limit_mb": 12676,
 "precision": "fp16 autocast + GradScaler (T4 has no fast bf16)",
 "head_stage_microbatch": 4,
 "partial_stage_microbatch": 8,
 "accumulation_steps_for_effective_batch_16": 2,
 "eval_microbatch": 16
}
=== SERIAL feature extraction (current wrapper)
  AST head warm-up                              B       0.05 h  [measured]  2 epochs x 4530 recordings, once per protocol
  AST partial fine-tuning                       B       0.55 h  [measured]  2 variant(s) x 8 epochs x 4530 recordings
  AST selection-validation each epoch           B       1.02 h  [measur

## 7. Verdict and `reports/r1/ast_probe.json`

In [9]:
problems = []
if not UNIT_TESTS_PASSED:
    problems.append("unit tests failed")
if WEIGHTS["revision"] != WEIGHTS["pinned_revision"]:
    problems.append("AST snapshot is not the pinned commit")
if WEIGHTS["recorded_sha256_matches"] is False:
    problems.append("AST weights SHA-256 differs from the recorded value")
if not LOADING["ok"]:
    problems.append(f"loading audit failed: {LOADING['non_head_new_or_reshaped']} {LOADING['unexpected']}")
if PREPROCESSOR["max_length"] != 1024 or PREPROCESSOR["num_mel_bins"] != 128 or PREPROCESSOR["sampling_rate"] != 16000:
    problems.append(f"AST preprocessing is not the expected 128-mel x 1024-frame grid at 16 kHz: {PREPROCESSOR}")
if not GRADIENT_PROOF["ok"]:
    problems.append("gradients/updates do not reach exactly the unfrozen groups")
if not all(p.get("loss_finite", True) for p in PROBES):
    problems.append("non-finite loss in a probe")

verdict = {"task": "R1-U2", "verified": not problems, "problems": problems, "pytest_summary": PYTEST_SUMMARY,
           "total_gpu_hours_with_reserve_serial": BUDGET_SERIAL["total_hours_with_reserve"],
           "total_gpu_hours_with_reserve_overlapped": BUDGET_OVERLAP["total_hours_with_reserve"],
           "oom_cases": [f"{p['stage']}/mb{p['microbatch']}" for p in PROBES if p["status"] == "oom"]}
evidence = {"schema_version": 1, "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), "environment": ENV, "source_sha256": SHA,
            "verdict": verdict, "weights": WEIGHTS, "preprocessor": PREPROCESSOR, "loading_audit": LOADING, "stage_reports": REPORTS,
            "gradient_proof": GRADIENT_PROOF, "training_probes": PROBES, "eval_probes": EVAL, "recommended": RECOMMENDED, "workload": WORKLOAD,
            "wavlm_clips_per_second_source": u1_reports[-1].name, "plan": DEFAULT_PLAN, "measured_inputs_serial": MEASURED_SERIAL,
            "budget_serial": BUDGET_SERIAL, "budget_overlapped": BUDGET_OVERLAP, "epoch_estimates": EPOCH}
out_dir = ROOT / "reports" / "r1"
target = out_dir / "ast_probe.json"
if target.exists():
    target.replace(out_dir / f"ast_probe.{time.strftime('%Y%m%dT%H%M%SZ', time.gmtime())}.json")   # keep the earlier run
target.write_text(json.dumps(evidence, indent=2, sort_keys=True, default=str), encoding="utf-8")
print(json.dumps(verdict, indent=2))
print("evidence written to:", target)
if WEIGHTS["recorded_sha256_matches"] != True:
    print("COPY into PRETRAINED['ast']['weights_sha256']:", WEIGHTS["sha256"])

{
  "task": "R1-U2",
  "verified": true,
  "problems": [],
  "pytest_summary": "15 passed in 21.20s",
  "total_gpu_hours_with_reserve_serial": 10.09,
  "total_gpu_hours_with_reserve_overlapped": 9.03,
  "oom_cases": []
}
evidence written to: /content/drive/MyDrive/PAI&AIES Project/reports/r1/ast_probe.json
